In [1]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [2]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
print("AWQ serving pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
AWQ serving pins installed


In [3]:
import os, signal, subprocess

PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for key, value in args.items():
        if value is None:
            cmd.append(key)
        else:
            cmd += [key, str(value)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 2054, logging to /content/server.log


In [4]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if response.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(
                        f"server healthy after about {waited}s: "
                        f"{url} -> 200"
                    )
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 168s: http://localhost:8000/v1/models -> 200


In [5]:
import json, urllib.request

with urllib.request.urlopen("http://localhost:8000/v1/models") as response:
    models = json.load(response)

print("served model:", models["data"][0]["id"])

served model: Qwen/Qwen2.5-1.5B-Instruct-AWQ


In [6]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader
!grep -iE "GPU blocks|AWQ|quantization" /content/server.log | tail -20

11723 MiB
INFO 09-02 13:01:11 api_server.py:713] args: Namespace(host=None, port=8000, uvicorn_log_level='info', allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=True, tool_call_parser='hermes', tool_parser_plugin='', model='Qwen/Qwen2.5-1.5B-Instruct-AWQ', task='auto', tokenizer=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path=None, download_dir=None, load_format='auto', config_format=<ConfigFormat.AUTO: 'auto'>, dtype='half', kv_cache_dtype='auto', quantizat

In [7]:
# Async A/B client for Lab W3D3 (engine swap).
# Paste the whole file as one Colab cell (after the vLLM server is healthy), then
# call run_sweep(...) as the day-3 README shows. It fires N concurrent chat
# completions per level with httpx + asyncio, excludes a warm-up round, and
# reports aggregate tokens/s at each concurrency level.
#
# It talks to the OpenAI-compatible /v1 endpoint, so the same client works
# against any team's service. No secrets: the local vLLM server needs no key.

import asyncio
import time

import httpx

# A fixed prompt set so every run measures the same work. Varied lengths, no
# duplicates. Requests cycle through this list.
FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

# Output lengths per request, cycled in order. This list is IDENTICAL to Monday's
# QUEUE in the day-2 lab, and it has to stay that way: the A/B is only honest if
# both engines are asked for exactly the same work. 24 requests, 18 that want 32
# tokens and 6 that want 256, so a long request is always in flight alongside
# short ones.
#
# The mixed lengths are the entire point. Ask every request for the same number
# of tokens and there is no straggler, static batching pays no tax, and
# continuous batching has nothing to win back. You would measure a flat speedup
# across concurrency and conclude, wrongly, that continuous batching does not
# scale.
QUEUE = [32, 32, 32, 256] * 6

# Fallback when a caller does not pass a length.
MAX_TOKENS = 128
# Warm-up requests per level, dropped from the timing.
WARMUP = 4


async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    """Fire one chat completion, return the count of completion tokens."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    # completion_tokens is what the server generated; fall back to counting.
    # Accounting note vs Monday: static_queue counted REQUESTED tokens, which
    # equals generated there (greedy decode runs to the cap). vLLM can stop at
    # EOS short of the cap, so counting usage is the honest number for it -
    # any bias this introduces runs AGAINST vLLM, never for it.
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct


async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    """Run total_requests requests, at most `concurrency` in flight at once."""
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    # Each request carries its own output length from QUEUE, so the workload
    # matches Monday's static-batching baseline request for request.
    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                         QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }


async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    """Sweep the concurrency levels; return a list of per-level result dicts.

    A warm-up round runs first and is discarded so model-load and cache-warm
    cost stays out of the measured numbers.
    """
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        # warm-up: fire WARMUP requests, ignore timing
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results


In [8]:
!curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8000/v1/models

200


In [9]:
!curl -s http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-1.5B-Instruct-AWQ","object":"model","created":1788354292,"owned_by":"vllm","root":"Qwen/Qwen2.5-1.5B-Instruct-AWQ","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-9b58043e72d14d92b3661c7b58c133ba","object":"model_permission","created":1788354292,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [10]:
import time
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
    timeout=180,
    max_retries=0,
)

prompt = "Explain how an inference server handles multiple requests in detail."

# Short warm-up, excluded from measurement.
client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=8,
    temperature=0,
)

start = time.perf_counter()
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128,
    temperature=0,
)
elapsed = time.perf_counter() - start

tokens = response.usage.completion_tokens
awq_tokens_per_s = tokens / elapsed

print("completion tokens:", tokens)
print("elapsed seconds:", round(elapsed, 2))
print("AWQ tokens/s:", round(awq_tokens_per_s, 1))

completion tokens: 128
elapsed seconds: 28.44
AWQ tokens/s: 4.5


In [11]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}],
        max_tokens=200,
    )
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-language=ru (Japan time zone ID 891)
```

Replace `[y

In [12]:
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")

    time.sleep(3)

    try:
        with urllib.request.urlopen(
            f"http://localhost:{port}/v1/models",
            timeout=2,
        ):
            print(f"WARNING: port {port} still answering")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

shutdown_server()

sent SIGTERM to process group of pid 2054
port 8000 is free


In [13]:
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

server = launch_server(SERVER_ARGS)
healthy = wait_for_health()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes
server pid 4019, logging to /content/server.log
server healthy after about 210s: http://localhost:8000/v1/models -> 200


In [14]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": p}],
        max_tokens=200,
    )
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests to execute model predictions and returning results to the client seamlessly and efficiently. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To find both the weather in Riyadh and the time in Tokyo, you would need to use two tool calls:

1. For the weather, you would make a tool call to "WeatherForecast".
2. For the time, you would make a tool call to "TimeIn" or "Time".

Here's an example of how you might structure the JSON for these two tool calls using a hypothetical JSON web API or a chatbot API provided by the service:

```json
{
    "tool": "WeatherForecast",
    "parameters": {
        "city": "Riyadh"
    },
    "_request_id": "123456789"
},
{
    "tool": "Time",
    "parameters": {
        "location": "Tokyo",
        "number_of_decimals": 2
    },
    "_request_id": "987654321"
}
```

In this example, I've included the API key pla

In [15]:
shutdown_server()

sent SIGTERM to process group of pid 4019
port 8000 is free


In [16]:
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

server = launch_server(SERVER_ARGS)
healthy = wait_for_health()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 5086, logging to /content/server.log
server healthy after about 129s: http://localhost:8000/v1/models -> 200


In [17]:
import urllib.request
import subprocess

url = "https://raw.githubusercontent.com/code2expert/ai-datacenter-bootcamp-labs/main/w3d4-quantise-and-lock/smoke_test.py"

req = urllib.request.urlopen(url)
exec(req.read().decode("utf-8"))

awq_result = run_smoke(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
)

print(awq_result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [19]:
shutdown_server()

sent SIGTERM to process group of pid 5086
port 8000 is free


In [20]:
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

server = launch_server(SERVER_ARGS)
healthy = wait_for_health()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes
server pid 6853, logging to /content/server.log
server healthy after about 132s: http://localhost:8000/v1/models -> 200


In [21]:
fp16_result = run_smoke(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
)

print(fp16_result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [24]:
import json

result = awq_result

with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

model_lock = (
    "# Model lock (team record)\n\n"
    "## The locked model\n\n"
    "- Model id: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`\n"
    "- Quantisation: `awq`\n"
    "- Why this one: The AWQ candidate passed the formal function-calling "
    "gate with 10/10, kept both distractor attempts call-free, and provides "
    "additional KV-cache capacity.\n\n"
    "## The launch flags\n\n"
    "- Flags: `--model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half "
    "--max-model-len 4096 --gpu-memory-utilization 0.85 "
    "--quantization awq --enable-auto-tool-choice "
    "--tool-call-parser hermes`\n"
    "- Tool-call parser: `hermes`\n\n"
    "## The smoke score\n\n"
    "- Score (valid behaviours out of 10): `10`\n"
    "- Distractor stayed call-free in the majority: `yes`\n"
    "- Passed the gate (>= 8/10 and distractor majority clean): `yes`\n"
    "- Measured against: AWQ `10/10`; FP16 `10/10`\n\n"
    "## Quality spot check note\n\n"
    "- FP16 was more coherent and complete overall. AWQ showed degradation "
    "on the tool-choice and quantisation prompts, but it passed the formal "
    "function-calling gate with valid tool behaviour and full distractor "
    "compliance.\n"
)

with open("model-lock.md", "w") as f:
    f.write(model_lock)

print(json.dumps(result, indent=2))
print("\nmodel-lock.md written")

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
  "total_attempts": 10,
  "score": 10,
  "distractor_attempts": 2,
  "distractor_call_free": 2,
  "distractor_majority_clean": true,
  "per_prompt": {
    "two_tool": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "single": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "distractor": {
      "k": 2,
      "wants_call": false,
      "valid": 0,
      "call_free": 2
    }
  },
  "passed": true
}

model-lock.md written


In [25]:
shutdown_server()

sent SIGTERM to process group of pid 6853
port 8000 is free


In [26]:
# Green-check verifier for Lab W3D4 (quantise and lock).
# Paste this as the last cell of your day-4 notebook and run it. It reads
# smoke_result.json (written from the smoke test) and model-lock.md, and checks
# that the smoke score meets the gate and that the lock file is fully filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) smoke result
    if not os.path.exists("smoke_result.json"):
        fail("smoke_result.json not found; write it in Cell 5")
    try:
        with open("smoke_result.json") as fh:
            result = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"smoke_result.json is not valid JSON: {exc}")

    for key in ("score", "total_attempts", "distractor_majority_clean", "passed"):
        if key not in result:
            fail(f"smoke_result.json missing key: {key}")

    score = result["score"]
    total = result["total_attempts"]
    if not isinstance(score, int) or not isinstance(total, int):
        fail("score and total_attempts must be integers")
    if total != 10:
        fail(f"total_attempts is {total}, the smoke test defines n=10")
    if score < 8:
        fail(f"smoke score {score}/10 is below the 8/10 gate")
    if not result["distractor_majority_clean"]:
        fail("distractor did not stay call-free in the majority; a model that "
             "always calls a tool fails the real consumer")
    if not result["passed"]:
        fail("smoke test reports passed=false")

    # 2) model-lock.md fully filled in
    if not os.path.exists("model-lock.md"):
        fail("model-lock.md not found")
    with open("model-lock.md") as fh:
        lock = fh.read()
    remaining = re.findall(r"FILL:", lock)
    if remaining:
        fail(f"model-lock.md has {len(remaining)} unfilled FILL: placeholders")
    # require a concrete model id line
    if not re.search(r"Model id:\s*\S+", lock):
        fail("model-lock.md has no concrete Model id")

    print(f"smoke score: {score}/{total}, distractor clean: "
          f"{result['distractor_majority_clean']}")
    print("model-lock.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
